# AMI-CPSD v1.0: Machine Learning & Deep Learning Benchmarks
## Multiclass Intrusion Detection and Zero-Day Attack Evaluation

This notebook trains baseline classifiers (Random Forest, Extra Trees, MLP) on **AMI-CPSD v1.0** data and evaluates zero-day attack detection performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

### 1. Load Data Splits

In [ ]:
data_dir = Path("../datasets")
df_train = pd.read_csv(data_dir / "train.csv", nrows=50000)
df_test = pd.read_csv(data_dir / "test.csv", nrows=10000)
df_zero_day = pd.read_csv(data_dir / "zero_day_test.csv", nrows=10000)

meta_cols = [
    "timestamp", "meter_id", "substation_id", "feeder_id", "transformer_id",
    "nan_id", "dc_id", "validator_group", "location", "consumer_type",
    "grid_state", "consensus_protocol", "attack_type", "attack_category",
    "attack_domain", "attack_severity", "attack_stage", "affected_features", "is_attack"
]
feat_cols = [c for c in df_train.columns if c not in meta_cols]

X_train, y_train = df_train[feat_cols], df_train["attack_type"].values
X_test, y_test = df_test[feat_cols], df_test["attack_type"].values
X_zd, y_zd = df_zero_day[feat_cols], df_zero_day["is_attack"].values

### 2. Preprocess & Scale Features

In [ ]:
imputer = SimpleImputer(strategy="mean")
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_scaled = scaler.transform(imputer.transform(X_test))
X_zd_scaled = scaler.transform(imputer.transform(X_zd))

### 3. Train Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)

preds = rf.predict(X_test_scaled)
print("Random Forest Standard Multiclass Classification Report:")
print(classification_report(y_test, preds, zero_division=0))

### 4. Zero-Day Intrusion Detection Evaluation

In [ ]:
zd_preds = rf.predict(X_zd_scaled)
zd_binary_preds = (zd_preds != 0).astype(int)

print("Zero-Day Binary Attack Detection Report:")
print(classification_report(y_zd, zd_binary_preds, target_names=["Normal", "Zero-Day Attack"]))